In [2]:
!pip install -q comet_ml
from comet_ml import Experiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.6/796.6 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 63.2 MB/s eta 0:00:00


In [3]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

In [4]:
X = np.load('X.npy')
y = np.load('y.npy')

In [18]:
class CFG:
    seed = 42
    test_size = 0.15
    val_size = 0.15
    batch_size = 256
    num_epochs = 100
    lr = 1e-3
    input_dim = X.shape[1]
    hidden_dims = [512, 256, 128]
    norm = 'batch'
    dropout = 0.3
    activation = 'relu'
    weight_decay = 0.0
    comet = True
    api = 'wX0NkEZpsfuU3t3cmf15A6HzP'
    project = 'oskelly-mlp'
    workspace = 'gp5_team1'
    num_workers = 2

In [6]:
def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
seed_everything(CFG.seed)

In [7]:
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=CFG.test_size, random_state=CFG.seed)

In [8]:
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=CFG.val_size / (1 - CFG.test_size), random_state=CFG.seed)

In [9]:
print(f'Train: {X_train.shape}')
print(f'Val: {X_val.shape}')
print(f'Test: {X_test.shape}')

Train: (6486, 588)
Val: (1391, 588)
Test: (1391, 588)


In [10]:
def make_loader(X, y, batch_size, shuffle=True):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32)
    dataset = TensorDataset(X_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=CFG.num_workers)

In [11]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)
print(f'X_train mean: {X_train_scaled.mean():.4f}  std: {X_train_scaled.std():.4f}')

X_train mean: -0.0000  std: 0.9983


In [12]:
train_loader = make_loader(X_train_scaled, y_train, CFG.batch_size, shuffle=True)
val_loader = make_loader(X_val_scaled, y_val, CFG.batch_size, shuffle=False)
test_loader = make_loader(X_test_scaled, y_test, CFG.batch_size, shuffle=False)
print(f'Батчей в train: {len(train_loader)}')
print(f'Батчей в val: {len(val_loader)}')
print(f'Батчей в test: {len(test_loader)}')

Батчей в train: 26
Батчей в val: 6
Батчей в test: 6


In [13]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, norm, dropout, activation):
        super(MLP, self).__init__()
        acts = {'relu': nn.ReLU, 'gelu': nn.GELU, 'tanh': nn.Tanh}
        layers = []
        prev = input_dim
        n = len(hidden_dims)
        for i, h in enumerate(hidden_dims):
            layers.append(nn.Linear(prev, h))
            if norm == 'batch':
                layers.append(nn.BatchNorm1d(h))
            elif norm == 'layer':
                layers.append(nn.LayerNorm(h))
            layers.append(acts[activation]())
            p = round(dropout * (n - i) / n, 4)
            if p > 0:
                layers.append(nn.Dropout(p))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x).squeeze(1)

In [14]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from tqdm import tqdm
def train_epoch(model, loader, optimizer, criterion, epoch, logger=None):
    model.train()
    total_loss = 0
    all_preds, all_targets = [], []
    for X_batch, y_batch in tqdm(loader):
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        all_preds.append(preds.detach().cpu())
        all_targets.append(y_batch.detach().cpu())
    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()
    preds_price = np.expm1(all_preds)
    target_price = np.expm1(all_targets)
    train_loss = total_loss / len(loader)
    r2 = r2_score(all_targets, all_preds)
    mae = mean_absolute_error(target_price, preds_price)
    rmse = mean_squared_error(target_price, preds_price) ** 0.5
    mape = np.mean(np.abs((target_price - preds_price) / target_price)) * 100
    print(f'Epoch {epoch}: train_loss = {train_loss:.4f}, r2 = {r2:.4f}, mae = {mae:.2f}, rmse = {rmse:.2f}, mape = {mape:.2f}%')
    if logger:
        logger.log_metrics({'train_loss': train_loss, 'train_r2': r2, 'train_mae': mae, 'train_rmse': rmse, 'train_mape': mape}, epoch=epoch)
    return train_loss, r2, mae, rmse, mape
def val_epoch(model, loader, criterion, epoch, logger=None):
    model.eval()
    total_loss = 0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for X_batch, y_batch in tqdm(loader):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            total_loss += loss.item()
            all_preds.append(preds.cpu())
            all_targets.append(y_batch.cpu())
    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()
    preds_price = np.expm1(all_preds)
    target_price = np.expm1(all_targets)
    val_loss = total_loss / len(loader)
    r2 = r2_score(all_targets, all_preds)
    mae = mean_absolute_error(target_price, preds_price)
    rmse = mean_squared_error(target_price, preds_price) ** 0.5
    mape = np.mean(np.abs((target_price - preds_price) / target_price)) * 100
    print(f'Epoch {epoch}: val_loss = {val_loss:.4f}, r2 = {r2:.4f}, mae = {mae:.2f}, rmse = {rmse:.2f}, mape = {mape:.2f}%')
    if logger:
        logger.log_metrics({'val_loss': val_loss, 'val_r2': r2, 'val_mae': mae, 'val_rmse': rmse, 'val_mape': mape}, epoch=epoch)
    return val_loss, r2, mae, rmse, mape

In [15]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = nn.MSELoss()
print(f'Устройство: {device}')

Устройство: cpu


In [16]:
def make_params(series, hidden_dims, norm, dropout, activation, optimizer, lr, weight_decay, scheduler):
    return {'series': series, 'hidden_dims': '-'.join(map(str, hidden_dims)), 'norm': norm, 'dropout': dropout, 'activation': activation, 'optimizer': optimizer, 'lr': lr, 'weight_decay': weight_decay, 'scheduler': scheduler, 'batch_size': CFG.batch_size, 'num_epochs': CFG.num_epochs, 'seed': CFG.seed, 'loss': 'MSELoss'}
def make_name(norm, dropout, hidden_dims, optimizer, lr, suffix=''):
    return f"mlp__norm-{norm}__d{dropout}__{'-'.join(map(str, hidden_dims))}__{optimizer}__lr{lr}{suffix}"
def run_experiment(run_name, model, optimizer, scheduler, params, num_epochs):
    model = model.to(device)
    best_val_loss = float('inf')
    ckpt = f'best_{run_name}.pth'
    logger = None
    if CFG.comet:
        logger = Experiment(api_key=CFG.api, project_name=CFG.project, workspace=CFG.workspace)
        logger.set_name(run_name)
        logger.log_parameters(params)
    for epoch in range(1, num_epochs + 1):
        train_epoch(model, train_loader, optimizer, criterion, epoch, logger)
        val_loss, val_r2, val_mae, val_rmse, val_mape = val_epoch(model, val_loader, criterion, epoch, logger)
        if scheduler is not None:
            scheduler.step(val_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), ckpt)
    model.load_state_dict(torch.load(ckpt))
    test_loss, test_r2, test_mae, test_rmse, test_mape = val_epoch(model, test_loader, criterion, epoch=0)
    if logger:
        logger.log_metrics({'test_loss': test_loss, 'test_r2': test_r2, 'test_mae': test_mae, 'test_rmse': test_rmse, 'test_mape': test_mape})
        logger.log_model(run_name, ckpt)
        logger.end()
    print(f'{run_name} | val {best_val_loss:.4f} | Test R2 {test_r2:.4f} | MAE {test_mae:,.0f}')
    return {'run': run_name, 'val_loss': best_val_loss, 'test_r2': test_r2, 'test_mae': test_mae, 'test_rmse': test_rmse, 'test_mape': test_mape, **params}
results = []

In [19]:
for norm in ['none', 'batch', 'layer']:
    seed_everything(CFG.seed)
    model = MLP(CFG.input_dim, CFG.hidden_dims, norm, CFG.dropout, CFG.activation)
    optimizer = optim.Adam(model.parameters(), lr=CFG.lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    params = make_params('norm', CFG.hidden_dims, norm, CFG.dropout, CFG.activation, 'adam', CFG.lr, 0.0, 'ReduceLROnPlateau')
    name = make_name(norm, CFG.dropout, CFG.hidden_dims, 'adam', CFG.lr)
    results.append(run_experiment(name, model, optimizer, scheduler, params, CFG.num_epochs))

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/gp5-team1/oskelly-mlp/6baf96a7ccb4425b88981085e9b0118f

100%|██████████| 26/26 [00:01<00:00, 16.93it/s]


Epoch 1: train_loss = 36.1390, r2 = -56.6488, mae = 215270096.00, rmse = 10964538820.87, mape = 491689.06%


100%|██████████| 6/6 [00:00<00:00, 16.37it/s]


Epoch 1: val_loss = 4.7428, r2 = -6.6929, mae = 3480371.00, rmse = 121609235.38, mape = 2732.42%


100%|██████████| 26/26 [00:01<00:00, 17.74it/s]


...

Epoch 100: val_loss = 0.1943, r2 = 0.6876, mae = 20302.51, rmse = 33634.70, mape = 35.27%


100%|██████████| 6/6 [00:00<00:00, 37.83it/s]


Epoch 0: val_loss = 0.2171, r2 = 0.6905, mae = 21305.95, rmse = 36443.60, mape = 37.72%


COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : mlp__norm-layer__d0.3__512-256-128__adam__lr0.001
COMET INFO:     url                   : https://www.comet.com/gp5-team1/oskelly-mlp/7fececad266d408e8c64fb39b3a49fc6
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     loss [260]       : (0.23900297284126282, 119.76481628417969)
COMET INFO:     test_loss        : 0.21705109377702078
COMET INFO:     test_mae         : 21305.947265625
COMET INFO:     test_mape        : 37.72147750854492
COMET INFO:     test_r2          : 0.6904543042182922
COMET INFO:     test_rmse        : 36443.599383156434
COMET INFO:     train_loss [100] : (0.29165728160968196, 47.02112036484938)
COMET INFO:     train_mae [10

mlp__norm-layer__d0.3__512-256-128__adam__lr0.001 | val 0.1891 | Test R2 0.6905 | MAE 21,306


In [20]:
for dropout in [0.0, 0.1, 0.3, 0.5]:
    seed_everything(CFG.seed)
    model = MLP(CFG.input_dim, CFG.hidden_dims, CFG.norm, dropout, CFG.activation)
    optimizer = optim.Adam(model.parameters(), lr=CFG.lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    params = make_params('dropout', CFG.hidden_dims, CFG.norm, dropout, CFG.activation, 'adam', CFG.lr, 0.0, 'ReduceLROnPlateau')
    name = make_name(CFG.norm, dropout, CFG.hidden_dims, 'adam', CFG.lr)
    results.append(run_experiment(name, model, optimizer, scheduler, params, CFG.num_epochs))

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/gp5-team1/oskelly-mlp/9d8e0ff9c79f4db190a49ade3a95b958

COMET INFO: Couldn't find a Git repository in '/content' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.
100%|██████████| 26/26 [00:00<00:00, 43.32it/s]


Epoch 1: train_loss = 100.2889, r2 = -156.4674, mae = 63933.46, rmse = 84705.64, mape = 100.00%


100%|██████████| 6/6 [00:00<00:00, 37.03it/s]


Epoch 1: val_loss = 89.4319, r2 = -145.1738, mae = 64284.55, rmse = 84068.89, mape = 99.99%


100%|██████████| 26/26 [00:00<00:00, 40.90it/s]


...

Epoch 100: val_loss = 0.2113, r2 = 0.6536, mae = 21439.05, rmse = 37204.07, mape = 36.70%


100%|██████████| 6/6 [00:00<00:00, 35.70it/s]


Epoch 0: val_loss = 0.2324, r2 = 0.6656, mae = 22184.87, rmse = 37445.78, mape = 40.05%


COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : mlp__norm-batch__d0.5__512-256-128__adam__lr0.001
COMET INFO:     url                   : https://www.comet.com/gp5-team1/oskelly-mlp/23c2181f1a844bdea66d438161678aa9
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     loss [260]       : (0.38607245683670044, 119.01220703125)
COMET INFO:     test_loss        : 0.23241252452135086
COMET INFO:     test_mae         : 22184.869140625
COMET INFO:     test_mape        : 40.051631927490234
COMET INFO:     test_r2          : 0.6656005382537842
COMET INFO:     test_rmse        : 37445.777332030375
COMET INFO:     train_loss [100] : (0.4856872226183231, 105.24379378098708)
COMET INFO:     train_mae [100]

mlp__norm-batch__d0.5__512-256-128__adam__lr0.001 | val 0.2002 | Test R2 0.6656 | MAE 22,185


In [21]:
for hidden_dims in [[256], [512, 256], [512, 256, 128], [1024, 512, 256]]:
    seed_everything(CFG.seed)
    model = MLP(CFG.input_dim, hidden_dims, CFG.norm, CFG.dropout, CFG.activation)
    optimizer = optim.Adam(model.parameters(), lr=CFG.lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    params = make_params('depth', hidden_dims, CFG.norm, CFG.dropout, CFG.activation, 'adam', CFG.lr, 0.0, 'ReduceLROnPlateau')
    name = make_name(CFG.norm, CFG.dropout, hidden_dims, 'adam', CFG.lr)
    results.append(run_experiment(name, model, optimizer, scheduler, params, CFG.num_epochs))

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Couldn't find a Git repository in '/content' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.
COMET INFO: Experiment is live on comet.com https://www.comet.com/gp5-team1/oskelly-mlp/8748e73887d446a5bd213edc3a9c3c27

100%|██████████| 26/26 [00:00<00:00, 35.17it/s]


Epoch 1: train_loss = 90.7359, r2 = -141.8202, mae = 63930.05, rmse = 84702.58, mape = 99.99%


100%|██████████| 6/6 [00:00<00:00, 17.65it/s]


Epoch 1: val_loss = 60.5540, r2 = -98.1764, mae = 64258.66, rmse = 84034.90, mape = 99.94%


100%|██████████| 26/26 [00:00<00:00, 41.22it/s]


...

Epoch 100: val_loss = 0.2053, r2 = 0.6689, mae = 20712.93, rmse = 36375.84, mape = 36.53%


100%|██████████| 6/6 [00:00<00:00, 28.53it/s]


Epoch 0: val_loss = 0.2481, r2 = 0.6523, mae = 22643.97, rmse = 39532.95, mape = 41.05%


COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : mlp__norm-batch__d0.3__1024-512-256__adam__lr0.001
COMET INFO:     url                   : https://www.comet.com/gp5-team1/oskelly-mlp/c39e346c9699450ab9bcfeca6e63c899
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     loss [260]       : (0.15824101865291595, 112.3584213256836)
COMET INFO:     test_loss        : 0.2480884144703547
COMET INFO:     test_mae         : 22643.96875
COMET INFO:     test_mape        : 41.04640579223633
COMET INFO:     test_r2          : 0.6522513628005981
COMET INFO:     test_rmse        : 39532.948486041365
COMET INFO:     train_loss [100] : (0.19170382275031164, 81.89398237375113)
COMET INFO:     train_mae [100]  :

mlp__norm-batch__d0.3__1024-512-256__adam__lr0.001 | val 0.2053 | Test R2 0.6523 | MAE 22,644


In [22]:
for opt_name in ['adam', 'adamw', 'sgd']:
    seed_everything(CFG.seed)
    model = MLP(CFG.input_dim, CFG.hidden_dims, CFG.norm, CFG.dropout, CFG.activation)
    if opt_name == 'adam':
        optimizer = optim.Adam(model.parameters(), lr=CFG.lr)
        wd = 0.0
    elif opt_name == 'adamw':
        optimizer = optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=1e-2)
        wd = 1e-2
    else:
        optimizer = optim.SGD(model.parameters(), lr=CFG.lr, momentum=0.9)
        wd = 0.0
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    params = make_params('optimizer', CFG.hidden_dims, CFG.norm, CFG.dropout, CFG.activation, opt_name, CFG.lr, wd, 'ReduceLROnPlateau')
    name = make_name(CFG.norm, CFG.dropout, CFG.hidden_dims, opt_name, CFG.lr)
    results.append(run_experiment(name, model, optimizer, scheduler, params, CFG.num_epochs))

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/gp5-team1/oskelly-mlp/06ffe3c2764a40ebb566b10333165ff5

COMET INFO: Couldn't find a Git repository in '/content' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.
100%|██████████| 26/26 [00:00<00:00, 34.69it/s]


Epoch 1: train_loss = 103.4489, r2 = -161.3928, mae = 63933.93, rmse = 84706.08, mape = 100.00%


100%|██████████| 6/6 [00:00<00:00, 36.43it/s]


Epoch 1: val_loss = 98.6894, r2 = -160.0692, mae = 64286.24, rmse = 84070.40, mape = 100.00%


100%|██████████| 26/26 [00:00<00:00, 34.38it/s]


...

Epoch 100: val_loss = 0.4018, r2 = 0.3544, mae = 27697.33, rmse = 44899.21, mape = 42.10%


100%|██████████| 6/6 [00:00<00:00, 37.40it/s]


Epoch 0: val_loss = 0.4124, r2 = 0.4001, mae = 29869.47, rmse = 47766.26, mape = 44.24%


COMET WARNING: Couldn't retrieve and log Google Colab notebook content, reason: 'NoneType' object is not subscriptable
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : mlp__norm-batch__d0.3__512-256-128__sgd__lr0.001
COMET INFO:     url                   : https://www.comet.com/gp5-team1/oskelly-mlp/21b786d207bb4dde8189823f88787a01
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     loss [260]       : (0.2932141125202179, 118.98240661621094)
COMET INFO:     test_loss        : 0.41243388255437213
COMET INFO:     test_mae         : 29869.466796875
COMET INFO:     test_mape        : 44.23845672607422
COMET INFO:     test_r2          : 0.4001436233520508
COMET INFO:     test_rmse        : 47766.2

mlp__norm-batch__d0.3__512-256-128__sgd__lr0.001 | val 0.3891 | Test R2 0.4001 | MAE 29,869


In [23]:
for lr in [1e-2, 1e-3, 1e-4]:
    seed_everything(CFG.seed)
    model = MLP(CFG.input_dim, CFG.hidden_dims, CFG.norm, CFG.dropout, CFG.activation)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    params = make_params('lr', CFG.hidden_dims, CFG.norm, CFG.dropout, CFG.activation, 'adam', lr, 0.0, 'ReduceLROnPlateau')
    name = make_name(CFG.norm, CFG.dropout, CFG.hidden_dims, 'adam', lr)
    results.append(run_experiment(name, model, optimizer, scheduler, params, CFG.num_epochs))
seed_everything(CFG.seed)
model = MLP(CFG.input_dim, CFG.hidden_dims, CFG.norm, CFG.dropout, CFG.activation)
optimizer = optim.Adam(model.parameters(), lr=CFG.lr)
params = make_params('lr', CFG.hidden_dims, CFG.norm, CFG.dropout, CFG.activation, 'adam', CFG.lr, 0.0, 'none')
name = make_name(CFG.norm, CFG.dropout, CFG.hidden_dims, 'adam', CFG.lr, '__noscheduler')
results.append(run_experiment(name, model, optimizer, None, params, CFG.num_epochs))

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/gp5-team1/oskelly-mlp/addec8e32c244c48a1d7c9b805b62da6

COMET INFO: Couldn't find a Git repository in '/content' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.
100%|██████████| 26/26 [00:00<00:00, 34.90it/s]


Epoch 1: train_loss = 32.0940, r2 = -50.1613, mae = 175696.27, rmse = 526226.20, mape = 421.07%


100%|██████████| 6/6 [00:00<00:00, 35.48it/s]


Epoch 1: val_loss = 9.5285, r2 = -14.3272, mae = 5545253.50, rmse = 49316738.12, mape = 15847.16%


100%|██████████| 26/26 [00:00<00:00, 34.66it/s]


...

Epoch 100: val_loss = 0.1943, r2 = 0.6826, mae = 20521.23, rmse = 34037.26, mape = 36.08%


100%|██████████| 6/6 [00:00<00:00, 36.22it/s]


Epoch 0: val_loss = 0.2338, r2 = 0.6701, mae = 22316.41, rmse = 37619.27, mape = 39.86%


COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : mlp__norm-batch__d0.3__512-256-128__adam__lr0.001__noscheduler
COMET INFO:     url                   : https://www.comet.com/gp5-team1/oskelly-mlp/c30e15b12c90456eac802ee47aecc42b
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     loss [260]       : (0.2557195723056793, 118.98240661621094)
COMET INFO:     test_loss        : 0.23380076636870703
COMET INFO:     test_mae         : 22316.40625
COMET INFO:     test_mape        : 39.86002731323242
COMET INFO:     test_r2          : 0.6700537204742432
COMET INFO:     test_rmse        : 37619.26825444642
COMET INFO:     train_loss [100] : (0.3151045785500453, 103.44890741201547)
COMET INFO:     train_

mlp__norm-batch__d0.3__512-256-128__adam__lr0.001__noscheduler | val 0.1943 | Test R2 0.6701 | MAE 22,316


In [24]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('test_r2', ascending=False).reset_index(drop=True)
results_df.to_csv('mlp_ablation_results.csv', index=False)
results_df[['series', 'run', 'val_loss', 'test_r2', 'test_mae', 'test_rmse', 'test_mape']]

,series,run,val_loss,test_r2,test_mae,test_rmse,test_mape
0,norm,mlp__norm-layer__d0.3__512-256-128__adam__lr0.001,0.189134,0.690454,21305.947266,36443.599383,37.721478
1,lr,mlp__norm-batch__d0.3__512-256-128__adam__lr0....,0.194269,0.670054,22316.406250,37619.268254,39.860027
2,dropout,mlp__norm-batch__d0.5__512-256-128__adam__lr0.001,0.200206,0.665601,22184.869141,37445.777332,40.051632
3,norm,mlp__norm-batch__d0.3__512-256-128__adam__lr0.001,0.192901,0.662468,22251.947266,37559.464107,40.055439
4,optimizer,mlp__norm-batch__d0.3__512-256-128__adam__lr0.001,0.192901,0.662468,22251.947266,37559.464107,40.055439
5,depth,mlp__norm-batch__d0.3__512-256-128__adam__lr0.001,0.192901,0.662468,22251.947266,37559.464107,40.055439
6,dropout,mlp__norm-batch__d0.3__512-256-128__adam__lr0.001,0.192901,0.662468,22251.947266,37559.464107,40.055439
7,lr,mlp__norm-batch__d0.3__512-256-128__adam__lr0.001,0.192901,0.662468,22251.947266,37559.464107,40.055439
8,lr,mlp__norm-batch__d0.3__512-256-128__adam__lr0.01,0.199426,0.653993,22721.152344,40146.896667,41.293640
9,depth,mlp__norm-batch__d0.3__1024-512-256__adam__lr0...,0.205251,0.652251,22643.968750,39532.948486,41.046406


In [ ]:
TOKEN = "ghp_твой_personal_access_token"
!git clone https://{TOKEN}@github.com/USER/REPO.git
%cd REPO

# 2. Создаём ветку
!git checkout -b feature/my-notebook

# 3. Кладём свой нотбук в папку репозитория
# (например, скопируй .ipynb из /content в текущую папку)
!cp /content/MyNotebook.ipynb .

# 4. Коммитим
!git config user.email "you@example.com"
!git config user.name "Your Name"
!git add MyNotebook.ipynb
!git commit -m "Add my notebook"
!git push origin feature/my-notebook

# 5. Сливаем в main локально и пушим
!git checkout main
!git merge feature/my-notebook
!git push origin main